# Jax Gradient Manipulations

## Lesson Goals:

By the end of this lesson you will understand gradient manipulations: how to stop gradient calculations from flowing backwards through parts of our graph, and how to stop skip applying gradients.

## Core Concepts:

- gradient stopping

- skipping gradient application

## Related Notebooks

- [Gradient Basics](./exe_07_grad_basics.ipynb) - basic derivatives
- [Advanced Autodiff](./exe_10_grad_advanced.ipynb) - custom derivatives

## Additional Reading:

- [Jax Documentation: Custom Derivative Rules](https://docs.jax.dev/en/latest/notebooks/Custom_derivative_rules_for_Python_code.html#summary)
- [Jax Documentation: Advanced Autodiff](https://docs.jax.dev/en/latest/advanced-autodiff.html#setup)

## Concepts In action:



In [ ]:
import jax
import jax.numpy as jnp

# Gradient Manipulations

## Gradient Stopping

### Example

Here we will cover gradient-stopping, which is useful when you want to skip backpropagating gradients through some subset of the computational graph. 

Note: this is akin to using `detach` in PyTorch to stop calculating the gradient. 

Let's pretend we have the following simple computation graph:

<img src="../assets/simple_graph.png" alt="drawing" width="400"/>

Say `r_2` should be some fixed relation - it was some "hard-crafted" feature specified by an external expert and we want to keep it. Unsurprisingly, the gradient will be different if we block it or allow the gradient to be calculated through it. Let's see that in action.

In [ ]:
import jax
import jax.numpy as jnp

def l_node(x, y):
    return (x - 0.5) * (y + 0.5)

def r_node(x, y):
    """
    # TODO: remove me
    """
    r_1 = x * y
    r_2 = x / y
    return r_1 + r_2

def r_node_blocked(x, y):
    """
    # TODO: remove me
    """
    r_1 = x * y
    r_2 = jax.lax.stop_gradient(x / y)
    return r_1 + r_2

def evaluation_fn(x, y, use_blocked_right):
    """
    """
    l_contrib = l_node(x, y)
    if use_blocked_right:
        r_contrib = r_node_blocked(x, y)
    else:
        r_contrib = r_node(x, y)
    return l_contrib + r_contrib

# Create some input values
x = 3.0
y = 4.0

grad_fn = jax.grad(evaluation_fn, argnums=(0, 1))
for should_block in [True, False]:
    dx, dy = grad_fn(x, y, should_block)
    print(f"{should_block=}\t{dx=}\t{dy=}")

### Real World Applications of Gradient Stopping

In principle, this is similar to:

- [Deep Deterministic Policy Gradients](https://spinningup.openai.com/en/latest/algorithms/ddpg.html#id1), where you have you pass your critic's prediction of the Q-value of the state to the actor, but do not want to update the actor's gradients with respect to the critic's prediction. 

- [Bootstrap Your Own Latents](https://arxiv.org/pdf/2006.07733)

<img src="../assets/byol.png" alt="drawing" width="800"/>

where you do self-supervised learning. The target is updated as an EMA of the online network.

## Skipping application of gradient

When you work with [Flax](https://flax.readthedocs.io/en/stable/) or [Equinox](https://github.com/patrick-kidger/equinox) you'll likely use the [optax](https://optax.readthedocs.io/en/latest/index.html) library to apply the gradient updates via algorithms such as adam, adagrad, etc. Let's consider a relatively simple example that we can build on our own below, using momentum.

### Example

Let's consider a simple optimizer that uses momentum to update our current gradient.

Note: for further reading or if you're interested in PyTorch check out [PyTorch Gradient Manipulation 1 - Ian Quah](https://ianq.ai/pytorch-gradients-pt1/)

In [ ]:
from dataclasses import dataclass

@dataclass
class MomentumOptimizer:
    decay: float
    
    def __post_init__(self):
        self.momentum = {}

    def update(
        self, 
        params: dict[str, jnp.ndarray],
        grads: dict[str, jnp.ndarray],
        learning_rate: float, 
        ignore: dict[str, bool] | None = None
    ) -> dict[str, jnp.ndarray]:
        """
        Accepts a params dict that has the same "shape" as grads and applies 
        momentum to gradient updates.

        Returns the updated params
        """
        if ignore is not None:
            assert set(ignore.keys()) == set(params.keys()) == set(grads.keys())
        else:
            assert set(params.keys()) == set(grads.keys())

        new_params = {}
        
        for k, v in params.items():
            should_ignore = ignore is not None and ignore.get(k, False)
            
            if not should_ignore:
                if k not in self.momentum:
                    self.momentum[k] = jnp.zeros_like(grads[k])
                
                # Update momentum: momentum = decay * prev_momentum + current_grad
                self.momentum[k] = self.decay * self.momentum[k] + grads[k]
                
                # Update parameters: param = param - learning_rate * momentum
                new_params[k] = v - learning_rate * self.momentum[k]
            else:
                # Don't update ignored parameters
                new_params[k] = v
    
        return new_params

In [ ]:
def test_simple_optimize():
    def _optimize_objective(params):
        return params['x']**2 + params['y']**2
    
    # Get gradient function
    optimize_objective = jax.value_and_grad(_optimize_objective)
    
    params = {
        'x': jnp.array(5.0),
        'y': jnp.array(-3.0)
    }
    
    optimizer = MomentumOptimizer(decay=0.9)
    learning_rate = 0.1
    
    print(f"{params=}")
    print(f"{_optimize_objective(params)=}")
    
    # Training loop
    for step in range(5):
        # Compute gradients
        evaluation, grads = optimize_objective(params)
        
        # Update parameters
        params = optimizer.update(params, grads, learning_rate)#, ignore={"x": False, "y": True if step % 2 == 0 else False})
        
        # Print progress
        _eval_res = float(evaluation)

        _x = float(params["x"])
        _y = float(params["y"])
        print(f"Step {step}: \t{_eval_res}\t{_x=}\t{_y=}")    

test_simple_optimize()